In [6]:
#!/usr/bin/env python3
"""
LPG Supply Chain Cost Decomposition ; Nigeria
=============================================
Reads first_step.gpkg and end_user_price.tif and produces
comprehensive cost decomposition plots and statistics.

Output folder: ./dataset/breakdown_plot/
"""

import os, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ============================================================
# USER CONFIGURABLE PATHS
# ============================================================
DATA_DIR = "dataset_250526_1138"
GPKG_PATH = os.path.join(DATA_DIR, "first_step.gpkg")
RESELL_LAYER = "resell"
END_USER_TIF = os.path.join(DATA_DIR, "end_user_price.tif")
URBAN_TIF = os.path.join(DATA_DIR, "Urban.tif")
POP_TIF = os.path.join(DATA_DIR, "Population.tif")
INCOME_TIF = os.path.join(DATA_DIR, "income_nigeria.tif")  # optional

OUT_DIR = os.path.join(DATA_DIR, "breakdown_plot")
os.makedirs(OUT_DIR, exist_ok=True)

# Urban threshold
URBAN_THRESHOLD = 20

# Cylinder weight (kg)
CYLINDER_WEIGHT_KG = 12.5

# Band indices (0-based) in end_user_price.tif
BAND_CAR_SHARE = 0
BAND_WALK_SHARE = 1
BAND_RESELLER_ID_WALK = 2
BAND_RESELLER_ID_CAR = 5
BAND_RES_COST_WALK = 8    # res_cost_kg_out_walk_ref
BAND_RES_COST_CAR = 9     # res_cost_kg_out_car_ref
BAND_COST_WALK = 10       # cost_kg_walk
BAND_COST_DRIVER = 11     # cost_kg_driver
BAND_LPG_USE = 16         # lpg_use_share
BAND_MEAN_COST = 18       # mean_user_cost

# ============================================================
# 1. LOAD DATA
# ============================================================
print("Loading data...")

resell = gpd.read_file(GPKG_PATH, layer=RESELL_LAYER)
print(f"Resellers loaded: {len(resell)}")

COST_COMPONENTS = [
    "cost_source",
    "cost_import_sea",
    "cost_sts",
    "cost_pre_bottling",
    "cost_import_land",
    "cost_border_wait",
    "cost_ferry",
    "cost_transport_to_storage",
    "cost_storage",
    "cost_storage_second",
    "cost_rebalancing",
    "cost_transport_from_storage",
    "cost_fil_plants",
    "cost_truck",
    "cost_res_shop",
]

for col in COST_COMPONENTS:
    if col not in resell.columns:
        resell[col] = 0.0
    else:
        resell[col] = pd.to_numeric(resell[col], errors='coerce').fillna(0.0)

resell_id_map = resell.set_index("id_res&fil")

with rasterio.open(END_USER_TIF) as src:
    end_user_arr = src.read()
    profile = src.profile
    transform = src.transform
    crs_raster = src.crs

nodata = profile.get("nodata", None)

def band_to_array(band_idx):
    arr = end_user_arr[band_idx].astype(np.float32)
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr

car_share    = band_to_array(BAND_CAR_SHARE)
walk_share   = band_to_array(BAND_WALK_SHARE)
res_cost_walk = band_to_array(BAND_RES_COST_WALK)
res_cost_car  = band_to_array(BAND_RES_COST_CAR)
cost_walk     = band_to_array(BAND_COST_WALK)
cost_driver   = band_to_array(BAND_COST_DRIVER)
lpg_use       = band_to_array(BAND_LPG_USE)
mean_user_cost= band_to_array(BAND_MEAN_COST)
walker_ids    = band_to_array(BAND_RESELLER_ID_WALK)
driver_ids    = band_to_array(BAND_RESELLER_ID_CAR)

collection_walk = cost_walk - res_cost_walk
collection_car  = cost_driver - res_cost_car

with rasterio.open(POP_TIF) as src:
    pop_arr = src.read(1).astype(np.float32)
    pop_nodata = src.nodata
    if pop_nodata is not None:
        pop_arr = np.where(pop_arr == pop_nodata, np.nan, pop_arr)
    pop_arr = np.where(pop_arr < 0, np.nan, pop_arr)

if pop_arr.shape != end_user_arr.shape[1:]:
    pop_aligned = np.full(end_user_arr.shape[1:], np.nan, dtype=np.float32)
    with rasterio.open(POP_TIF) as src_pop:
        reproject(
            source=rasterio.band(src_pop, 1),
            destination=pop_aligned,
            src_transform=src_pop.transform,
            src_crs=src_pop.crs,
            dst_transform=transform,
            dst_crs=crs_raster,
            dst_nodata=np.nan,
            resampling=Resampling.nearest)
    pop_arr = pop_aligned

with rasterio.open(URBAN_TIF) as src:
    urban_arr = src.read(1).astype(np.float32)
    urban_nodata = src.nodata
    if urban_nodata is not None:
        urban_arr = np.where(urban_arr == urban_nodata, np.nan, urban_arr)

if urban_arr.shape != end_user_arr.shape[1:]:
    urban_aligned = np.full(end_user_arr.shape[1:], np.nan, dtype=np.float32)
    with rasterio.open(URBAN_TIF) as src_u:
        reproject(
            source=rasterio.band(src_u, 1),
            destination=urban_aligned,
            src_transform=src_u.transform,
            src_crs=src_u.crs,
            dst_transform=transform,
            dst_crs=crs_raster,
            dst_nodata=np.nan,
            resampling=Resampling.nearest)
    urban_arr = urban_aligned

urban_flag = urban_arr >= URBAN_THRESHOLD

# ============================================================
# 2. MASKS AND WEIGHTS
# ============================================================
valid_mask = (
    np.isfinite(pop_arr) & (pop_arr > 0) &
    np.isfinite(lpg_use) & (lpg_use > 0) &
    np.isfinite(mean_user_cost)
)

weights = pop_arr * lpg_use
urban_mask = valid_mask & urban_flag
rural_mask = valid_mask & ~urban_flag

def weighted_mean(arr, mask=None):
    if mask is None:
        mask = valid_mask
    w = weights[mask]
    vals = arr[mask]
    fin = np.isfinite(vals)
    if fin.sum() == 0:
        return np.nan
    return np.average(vals[fin], weights=w[fin])

def save_fig(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved {path}")

# ============================================================
# 3. PRE‑COMPUTE AVERAGES
# ============================================================
mean_upstream_walk = weighted_mean(res_cost_walk)
mean_coll_walk     = weighted_mean(collection_walk)
mean_total_walk    = weighted_mean(cost_walk)

mean_upstream_car  = weighted_mean(res_cost_car)
mean_coll_car      = weighted_mean(collection_car)
mean_total_car     = weighted_mean(cost_driver)

avg_resell_costs = resell[COST_COMPONENTS].mean()

up_walk_urb = weighted_mean(res_cost_walk, urban_mask)
coll_walk_urb = weighted_mean(collection_walk, urban_mask)
up_walk_rur = weighted_mean(res_cost_walk, rural_mask)
coll_walk_rur = weighted_mean(collection_walk, rural_mask)

up_car_urb = weighted_mean(res_cost_car, urban_mask)
coll_car_urb = weighted_mean(collection_car, urban_mask)
up_car_rur = weighted_mean(res_cost_car, rural_mask)
coll_car_rur = weighted_mean(collection_car, rural_mask)

up_walk_all   = mean_upstream_walk
coll_walk_all = mean_coll_walk
up_car_all    = mean_upstream_car
coll_car_all  = mean_coll_car

walk_share_all = weighted_mean(walk_share)
car_share_all  = 1 - walk_share_all
up_mean_all   = walk_share_all * up_walk_all + car_share_all * up_car_all
coll_mean_all = walk_share_all * coll_walk_all + car_share_all * coll_car_all

comp_frac = (avg_resell_costs / avg_resell_costs.sum()).to_dict()

total_pop_urb = weights[urban_mask].sum()
total_pop_rur = weights[rural_mask].sum()

walk_share_urb = weighted_mean(walk_share, urban_mask)
car_share_urb = 1 - walk_share_urb
walk_share_rur = weighted_mean(walk_share, rural_mask)
car_share_rur = 1 - walk_share_rur

up_mean_urb = walk_share_urb * up_walk_urb + car_share_urb * up_car_urb
coll_mean_urb = walk_share_urb * coll_walk_urb + car_share_urb * coll_car_urb
up_mean_rur = walk_share_rur * up_walk_rur + car_share_rur * up_car_rur
coll_mean_rur = walk_share_rur * coll_walk_rur + car_share_rur * coll_car_rur

# Global consistent pie chart colors
colors_dict = {comp: plt.cm.tab20(i) for i, comp in enumerate(COST_COMPONENTS)}
colors_dict['Collection'] = 'orange'

# ============================================================
# 4. PLOTS
# ============================================================
print("\nCreating plots...")

# -----------------------------------------------------------------
# PLOT 01: Average upstream waterfall (narrowed, large legend)
# -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5, 8)) # Decreased horizontal extension
comp_vals = avg_resell_costs.values
cumulative = np.cumsum(comp_vals)
for i, (comp, val) in enumerate(avg_resell_costs.items()):
    ax.bar(0, val, bottom=cumulative[i] - val, color=colors_dict[comp], edgecolor='black', label=comp)
ax.set_xticks([])
ax.set_ylabel("Cost (USD/kg)")
ax.set_title("Average Upstream Cost\n(Source → Reseller)")
ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), 
          fontsize=11, handlelength=2, handleheight=2)
save_fig(fig, "01_average_upstream_waterfall.png")

# -----------------------------------------------------------------
# PLOT 02: Average upstream waterfall (zoomed in y-axis)
# -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5, 8))
for i, (comp, val) in enumerate(avg_resell_costs.items()):
    ax.bar(0, val, bottom=cumulative[i] - val, color=colors_dict[comp], edgecolor='black', label=comp)
ax.set_xticks([])
ax.set_ylabel("Cost (USD/kg)")
# Cut the axis to emphasize the top components (note: applied to Y-axis since cost is vertical)
ax.set_ylim(bottom=0.68)
ax.set_title("Average Upstream Cost\n(Zoomed > 0.68 USD)")
ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), 
          fontsize=11, handlelength=2, handleheight=2)
save_fig(fig, "02_average_upstream_waterfall_zoomed.png")

# -----------------------------------------------------------------
# PLOT 03: End-user cost decomposition
# -----------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.bar([0], mean_upstream_walk, label='Upstream')
ax1.bar([0], mean_coll_walk, bottom=mean_upstream_walk, label='Collection')
ax1.set_title("Walker ; weighted average")
ax1.set_ylabel("USD/kg")
ax1.legend()
ax2.bar([0], mean_upstream_car, label='Upstream')
ax2.bar([0], mean_coll_car, bottom=mean_upstream_car, label='Collection')
ax2.set_title("Driver ; weighted average")
ax2.set_ylabel("USD/kg")
ax2.legend()
fig.suptitle("Average End-User Cost Decomposition")
save_fig(fig, "03_enduser_cost_decomposition.png")

# -----------------------------------------------------------------
# PLOT 04: Simple total cost waterfall
# -----------------------------------------------------------------
fig, ax = plt.subplots()
up_avg = (mean_upstream_walk + mean_upstream_car) / 2
coll_avg = (mean_coll_walk + mean_coll_car) / 2
ax.bar(0, up_avg, label='Upstream (avg)', color='steelblue')
ax.bar(0, coll_avg, bottom=up_avg, label='Collection (avg)', color='orange')
ax.set_ylabel("USD/kg")
ax.set_title("Average Total End-User Cost")
ax.legend()
save_fig(fig, "04_total_cost_waterfall_simple.png")

# -----------------------------------------------------------------
# PLOT 05: Histogram of walk and drive costs
# -----------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax_, data, label in zip([ax1, ax2], [cost_walk, cost_driver], ['Walk', 'Drive']):
    d = data[valid_mask].flatten()
    d = d[np.isfinite(d)]
    ax_.hist(d, bins=100, color='gray', alpha=0.7)
    ax_.axvline(np.nanmedian(d), color='red', linestyle='--', label='Median')
    ax_.set_title(f"{label} End-User Cost")
    ax_.set_xlabel("USD/kg")
    ax_.legend()
save_fig(fig, "05_cost_distribution_walk_drive.png")

# -----------------------------------------------------------------
# PLOT 06: Hexbin: collection cost vs upstream cost
# -----------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax_, coll, up, title in zip([ax1, ax2],
                                [collection_walk, collection_car],
                                [res_cost_walk, res_cost_car],
                                ['Walk', 'Drive']):
    m = valid_mask & np.isfinite(coll) & np.isfinite(up)
    hb = ax_.hexbin(up[m].flatten(), coll[m].flatten(), gridsize=50,
                    cmap='viridis', mincnt=1, norm=mcolors.LogNorm())
    plt.colorbar(hb, ax=ax_, label='Count (log)')
    ax_.set_xlabel("Upstream cost (USD/kg)")
    ax_.set_ylabel("Collection cost (USD/kg)")
    ax_.set_title(title)
save_fig(fig, "06_hexbin_collection_vs_upstream.png")

# -----------------------------------------------------------------
# PLOT 07: Hexbin: collection vs upstream (Shared Y-Axis)
# -----------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax_, coll, up, title in zip([ax1, ax2],
                                [collection_walk, collection_car],
                                [res_cost_walk, res_cost_car],
                                ['Walk (Shared Y)', 'Drive (Shared Y)']):
    m = valid_mask & np.isfinite(coll) & np.isfinite(up)
    hb = ax_.hexbin(up[m].flatten(), coll[m].flatten(), gridsize=50,
                    cmap='viridis', mincnt=1, norm=mcolors.LogNorm())
    plt.colorbar(hb, ax=ax_, label='Count (log)')
    ax_.set_xlabel("Upstream cost (USD/kg)")
    if ax_ == ax1:
        ax_.set_ylabel("Collection cost (USD/kg)")
    ax_.set_title(title)
save_fig(fig, "07_hexbin_collection_vs_upstream_sharedy.png")

# -----------------------------------------------------------------
# PLOT 08: Urban vs rural driver stacked bar
# -----------------------------------------------------------------
fig, ax = plt.subplots()
labels = ['Urban', 'Rural']
ax.bar(labels, [up_car_urb, up_car_rur], label='Upstream')
ax.bar(labels, [coll_car_urb, coll_car_rur], bottom=[up_car_urb, up_car_rur], label='Collection')
ax.set_ylabel("USD/kg")
ax.set_title("Driver Cost Decomposition ; Urban vs Rural")
ax.legend()
save_fig(fig, "08_urban_rural_driver.png")

# -----------------------------------------------------------------
# PLOT 09: Map of collection cost share (Independent dynamic scales)
# -----------------------------------------------------------------
share_walk = collection_walk / cost_walk
share_car  = collection_car / cost_driver
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# Removing forced vmin/vmax so each maps beautifully to its natural data range
im1 = ax1.imshow(share_walk, cmap='coolwarm')
plt.colorbar(im1, ax=ax1, fraction=0.046)
ax1.set_title("Collection cost share ; Walk")
ax1.axis('off')

im2 = ax2.imshow(share_car, cmap='coolwarm')
plt.colorbar(im2, ax=ax2, fraction=0.046)
ax2.set_title("Collection cost share ; Drive")
ax2.axis('off')
save_fig(fig, "09_map_collection_share.png")

# -----------------------------------------------------------------
# PLOT 10: Sensitivity to vehicle cost per km
# -----------------------------------------------------------------
veh_factors = [0.5, 1.0, 2.0]
veh_means = []
for vf in veh_factors:
    new_car = res_cost_car + (collection_car * vf)
    new_mean = np.where(walk_share > car_share, cost_walk, new_car)
    veh_means.append(weighted_mean(new_mean))
fig, ax = plt.subplots()
ax.plot(veh_factors, veh_means, marker='s')
ax.set_xlabel("Vehicle cost multiplier")
ax.set_ylabel("Mean end-user cost (USD/kg)")
ax.set_title("Sensitivity to Vehicle Operating Cost")
save_fig(fig, "10_sensitivity_vehicle_cost.png")

# -----------------------------------------------------------------
# PLOT 11: Representative pixel analysis (Stacked Vertical Bar)
# -----------------------------------------------------------------
pop_flat = pop_arr.copy()
pop_flat[~valid_mask] = np.inf
idx = np.nanargmin(np.abs(pop_flat - np.nanmedian(pop_arr[valid_mask])))
row, col = np.unravel_index(idx, pop_arr.shape)

walk_id = int(walker_ids[row, col]) if not np.isnan(walker_ids[row, col]) else None
if walk_id is not None and walk_id in resell_id_map.index:
    row_res = resell_id_map.loc[walk_id]
    comps = row_res[COST_COMPONENTS].to_dict()
    
    fig, ax = plt.subplots(figsize=(5, 8))
    cumul = 0
    for comp, val in comps.items():
        if val > 0:
            ax.bar("Upstream", val, bottom=cumul, color=colors_dict[comp], edgecolor='black', label=comp)
            cumul += val
    
    # Visual confirmation that parts sum to the total pixel cost
    total_cost = res_cost_walk[row, col]
    ax.axhline(total_cost, color='red', linestyle='--', linewidth=2, label=f'Total Raster Cost: {total_cost:.3f}')
    
    ax.set_ylabel("USD/kg")
    ax.set_title("Upstream Stack at Representative Pixel")
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), fontsize=10)
    save_fig(fig, "11_pixel_upstream_breakdown.png")

# -----------------------------------------------------------------
# PLOT 12: Map of walk and drive costs side by side
# -----------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for ax_, arr, title in zip([ax1, ax2], [cost_walk, cost_driver], ['Walk Cost','Drive Cost']):
    im = ax_.imshow(arr, cmap='inferno')
    plt.colorbar(im, ax=ax_, fraction=0.046)
    ax_.set_title(title)
    ax_.axis('off')
save_fig(fig, "12_map_walk_drive_cost.png")

# -----------------------------------------------------------------
# PLOT 13: Map of car share
# -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(car_share, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Car share')
ax.set_title("Car Share")
ax.axis('off')
save_fig(fig, "13_map_car_share.png")

# -----------------------------------------------------------------
# PLOTS 14‑22: Decomposed pie charts (with High Contrast Colors)
# -----------------------------------------------------------------
def make_decomposed_pie(up_value, coll_value, title, filename):
    slices = {comp: frac * up_value for comp, frac in comp_frac.items()}
    slices['Collection'] = coll_value
    slices = {k: v for k, v in slices.items() if v > 1e-6}

    # Match high contrast colors mapped at the global level
    pie_colors = [colors_dict.get(k, 'gray') for k in slices.keys()]

    fig, ax = plt.subplots(figsize=(9, 9))
    wedges, texts, autotexts = ax.pie(
        slices.values(),
        labels=None,
        colors=pie_colors,
        autopct='%1.1f%%',
        startangle=140,
        pctdistance=0.85
    )
    for t in autotexts:
        t.set_fontsize(9)

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(wedges, slices.keys(), title="Components",
              loc="center left", bbox_to_anchor=(1, 0, 0.5, 1), fontsize=8)
    save_fig(fig, filename)

make_decomposed_pie(up_mean_all, coll_mean_all, "Mean User ; Overall", "14_pie_mean_all.png")
make_decomposed_pie(up_mean_urb, coll_mean_urb, "Mean User ; Urban",  "15_pie_mean_urban.png")
make_decomposed_pie(up_mean_rur, coll_mean_rur, "Mean User ; Rural",  "16_pie_mean_rural.png")

make_decomposed_pie(up_walk_all, coll_walk_all, "Walker ; Overall", "17_pie_walk_all.png")
make_decomposed_pie(up_walk_urb, coll_walk_urb, "Walker ; Urban",  "18_pie_walk_urban.png")
make_decomposed_pie(up_walk_rur, coll_walk_rur, "Walker ; Rural",  "19_pie_walk_rural.png")

make_decomposed_pie(up_car_all, coll_car_all, "Driver ; Overall", "20_pie_drive_all.png")
make_decomposed_pie(up_car_urb, coll_car_urb, "Driver ; Urban",  "21_pie_drive_urban.png")
make_decomposed_pie(up_car_rur, coll_car_rur, "Driver ; Rural",  "22_pie_drive_rural.png")


print("\nAll plots saved to", OUT_DIR)

Loading data...
Resellers loaded: 2413

Creating plots...
  Saved dataset_250526_1138\breakdown_plot\01_average_upstream_waterfall.png
  Saved dataset_250526_1138\breakdown_plot\02_average_upstream_waterfall_zoomed.png
  Saved dataset_250526_1138\breakdown_plot\03_enduser_cost_decomposition.png
  Saved dataset_250526_1138\breakdown_plot\04_total_cost_waterfall_simple.png
  Saved dataset_250526_1138\breakdown_plot\05_cost_distribution_walk_drive.png
  Saved dataset_250526_1138\breakdown_plot\06_hexbin_collection_vs_upstream.png
  Saved dataset_250526_1138\breakdown_plot\07_hexbin_collection_vs_upstream_sharedy.png
  Saved dataset_250526_1138\breakdown_plot\08_urban_rural_driver.png
  Saved dataset_250526_1138\breakdown_plot\09_map_collection_share.png
  Saved dataset_250526_1138\breakdown_plot\10_sensitivity_vehicle_cost.png
  Saved dataset_250526_1138\breakdown_plot\11_pixel_upstream_breakdown.png
  Saved dataset_250526_1138\breakdown_plot\12_map_walk_drive_cost.png
  Saved dataset_250

In [ ]:
#!/usr/bin/env python3
"""
Deep consistency check for LPG cost chain
-----------------------------------------
1. Internal component sums (storage / filling / resell)
2. Cross‑layer: resell LPG_price = filling LPG_price + truck + shop
3. Raster;vector mapping:
   - missing reseller IDs in raster (compared to resell vector)
   - reference cost accuracy (only for IDs that exist)
   - filling cost accuracy
4. Clear final summary with pass/fail for each check.
"""

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

# ────────────────────────────────────────────────────────
# Configuration
# ────────────────────────────────────────────────────────
DATA_DIR = "dataset"
GPKG_PATH = os.path.join(DATA_DIR, "first_step.gpkg")
TIF_PATH  = os.path.join(DATA_DIR, "end_user_price.tif")

# ────────────────────────────────────────────────────────
# 1. Load vector layers
# ────────────────────────────────────────────────────────
print("=" * 60)
print("LOADING DATA")
print("=" * 60)
ps   = gpd.read_file(GPKG_PATH, layer="primary_storage_allocated")
fill = gpd.read_file(GPKG_PATH, layer="filling_points")
resell = gpd.read_file(GPKG_PATH, layer="resell")

print(f"primary_storage_allocated : {len(ps):,} rows")
print(f"filling_points            : {len(fill):,} rows")
print(f"resell                    : {len(resell):,} rows\n")

# ────────────────────────────────────────────────────────
# 2. Internal consistency: component sums == LPG_price
# ────────────────────────────────────────────────────────
print("=" * 60)
print("1. INTERNAL COMPONENT SUMS")
print("=" * 60)

ps_components = [
    "cost_source", "cost_import_sea", "cost_sts", "cost_pre_bottling",
    "cost_import_land", "cost_border_wait", "cost_ferry",
    "cost_transport_to_storage", "cost_storage", "cost_storage_second",
    "cost_rebalancing"
]

fill_components = ps_components + ["cost_transport_from_storage", "cost_fil_plants"]

resell_components = fill_components + ["cost_truck", "cost_res_shop"]

def check_sum(layer_name, gdf, components, id_col="id_supply"):
    if "LPG_price" not in gdf.columns:
        return False, f"Missing LPG_price column"
    missing = [c for c in components if c not in gdf.columns]
    if missing:
        return False, f"Missing columns: {missing}"

    gdf = gdf.copy()
    for c in components:
        gdf[c] = pd.to_numeric(gdf[c], errors='coerce').fillna(0.0)
    gdf["LPG_price"] = pd.to_numeric(gdf["LPG_price"], errors='coerce')
    gdf["sum"] = gdf[components].sum(axis=1)
    diff = (gdf["LPG_price"] - gdf["sum"]).abs()
    bad = diff > 1e-6
    if bad.any():
        msg = f"{bad.sum()} rows out of {len(gdf)} have mismatched sum"
        return False, msg
    return True, f"All {len(gdf)} rows consistent"

results = {}
for name, gdf, comps in [
    ("primary_storage_allocated", ps, ps_components),
    ("filling_points", fill, fill_components),
    ("resell", resell, resell_components)
]:
    ok, msg = check_sum(name, gdf, comps)
    results[f"internal_{name}"] = ok
    print(f"{'✅' if ok else '❌'} {name}: {msg}")

# ────────────────────────────────────────────────────────
# 3. Cross‑layer: resell LPG_price = filling LPG_price + truck + shop
# ────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. CROSS‑LAYER: RESELL → FILLING")
print("=" * 60)

fill_prices = fill.set_index("id_res&fil")["LPG_price"].to_dict()

resell["filling_lpg"] = resell["filling_reference"].map(fill_prices)
resell["expected_lpg"] = resell["filling_lpg"] + resell["cost_truck"] + resell["cost_res_shop"]
diff_cross = (resell["LPG_price"] - resell["expected_lpg"]).abs()
bad_cross = diff_cross > 1e-6
n_bad = bad_cross.sum()
results["cross_resell_filling"] = (n_bad == 0)
if n_bad == 0:
    print("✅ Resell LPG_price = filling LPG_price + cost_truck + cost_res_shop (all rows)")
else:
    print(f"❌ {n_bad} rows break the equation")
    # print first few
    for idx in resell[bad_cross].index[:5]:
        row = resell.loc[idx]
        print(f"   ID {row['id_res&fil']}: LPG_price={row['LPG_price']:.6f}, "
              f"expected={row['expected_lpg']:.6f}")

# (Note: The old cost_res_kg_in / cost_res_kg_out columns are ignored entirely)

# ────────────────────────────────────────────────────────
# 4. Raster;vector consistency
# ────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. RASTER vs VECTOR CONSISTENCY")
print("=" * 60)

if not os.path.exists(TIF_PATH):
    print("⚠️  Raster file not found. Skipping.")
    results["raster"] = False
else:
    with rasterio.open(TIF_PATH) as src:
        nodata = src.nodata
        bands = src.read()
        # Bands (0‑based index):
        # 2 = walk_id, 5 = car_id,
        # 8 = res_cost_walk_ref, 9 = res_cost_car_ref,
        # 12 = filling_id_walk, 13 = filling_id_car,
        # 14 = cost_fil_kg_out_walk_ref, 15 = cost_fil_kg_out_car_ref
        walk_id = bands[2].astype(np.float32)
        car_id  = bands[5].astype(np.float32)
        res_walk_ref = bands[8].astype(np.float32)
        res_car_ref  = bands[9].astype(np.float32)
        fill_id_walk = bands[12].astype(np.float32)
        fill_id_car  = bands[13].astype(np.float32)
        fill_cost_walk = bands[14].astype(np.float32)
        fill_cost_car  = bands[15].astype(np.float32)

    if nodata is not None:
        for arr in [walk_id, car_id, res_walk_ref, res_car_ref,
                    fill_id_walk, fill_id_car, fill_cost_walk, fill_cost_car]:
            arr[arr == nodata] = np.nan

    # 4a. Unique reseller IDs in raster vs. resell vector
    resell_ids_set = set(resell["id_res&fil"].unique())
    walk_ids_uniq = set(int(x) for x in np.unique(walk_id[~np.isnan(walk_id)]))
    car_ids_uniq  = set(int(x) for x in np.unique(car_id[~np.isnan(car_id)]))
    missing_walk = walk_ids_uniq - resell_ids_set
    missing_car  = car_ids_uniq - resell_ids_set

    results["raster_missing_walk"] = (len(missing_walk) == 0)
    results["raster_missing_car"]  = (len(missing_car) == 0)

    if missing_walk:
        print(f"❌ {len(missing_walk)} walk reseller IDs not in resell layer "
              f"(first 10: {sorted(list(missing_walk))[:10]})")
    else:
        print("✅ All walk reseller IDs present in resell layer")

    if missing_car:
        print(f"❌ {len(missing_car)} car reseller IDs not in resell layer "
              f"(first 10: {sorted(list(missing_car))[:10]})")
    else:
        print("✅ All car reseller IDs present in resell layer")

    # 4b. Reference cost accuracy (only for IDs that exist)
    resell_price_map = resell.set_index("id_res&fil")["LPG_price"].to_dict()

    def sample_check(id_arr, ref_arr, name, resell_price_map, missing_set):
        valid_mask = np.isfinite(id_arr) & np.isfinite(ref_arr)
        idx = np.argwhere(valid_mask)
        if len(idx) == 0:
            print(f"   ⚠️  No valid {name} pixels to sample.")
            return 0, 0, 0  # total, missing_id_mismatches, true_mismatches
        rng = np.random.default_rng(42)
        sample_size = min(10000, len(idx))
        sample_idx = rng.choice(len(idx), size=sample_size, replace=False)
        missing_mismatches = 0
        price_mismatches = 0
        for i in sample_idx:
            r, c = idx[i]
            rid = int(id_arr[r, c])
            ref_cost = ref_arr[r, c]
            if rid in missing_set:
                missing_mismatches += 1
                continue
            actual = resell_price_map.get(rid, None)
            if actual is None:
                missing_mismatches += 1  # shouldn't happen if not in missing_set, but safety
                continue
            if abs(ref_cost - actual) > 1e-5:
                price_mismatches += 1
        total_sample = sample_size
        return total_sample, missing_mismatches, price_mismatches

    for mode, id_arr, ref_arr, miss_set in [
        ("walk", walk_id, res_walk_ref, missing_walk),
        ("car",  car_id,  res_car_ref,  missing_car)
    ]:
        total, missing_mm, price_mm = sample_check(id_arr, ref_arr, mode,
                                                   resell_price_map, miss_set)
        print(f"\n   ── {mode} reference cost sampling (n={total}) ──")
        if missing_mm > 0:
            print(f"   ⚠️  {missing_mm} pixels reference IDs missing from resell layer (expected)")
        if price_mm == 0:
            print(f"   ✅ All remaining pixels match resell LPG_price")
        else:
            print(f"   ❌ {price_mm} pixels have a true price mismatch")
        results[f"raster_{mode}_price_match"] = (price_mm == 0)

    # 4c. Filling point cost references
    filling_cost_map = fill.set_index("id_res&fil")["cost_fil_kg_out"].to_dict()
    def check_filling(id_arr, cost_arr, name):
        valid = np.isfinite(id_arr) & np.isfinite(cost_arr)
        idx = np.argwhere(valid)
        if len(idx) == 0:
            print(f"   ⚠️  No valid filling {name} pixels")
            return True, 0
        rng = np.random.default_rng(42)
        sample_size = min(10000, len(idx))
        sample_idx = rng.choice(len(idx), size=sample_size, replace=False)
        mismatches = 0
        for i in sample_idx:
            r, c = idx[i]
            fid = int(id_arr[r, c])
            rcost = cost_arr[r, c]
            actual = filling_cost_map.get(fid)
            if actual is None or abs(rcost - actual) > 1e-5:
                mismatches += 1
        ok = (mismatches == 0)
        return ok, mismatches

    ok_fw, mm_fw = check_filling(fill_id_walk, fill_cost_walk, "walk")
    ok_fc, mm_fc = check_filling(fill_id_car, fill_cost_car, "car")
    results["raster_filling_walk"] = ok_fw
    results["raster_filling_car"] = ok_fc
    print(f"\n   ── filling cost references ──")
    print(f"   {'✅' if ok_fw else '❌'} walk: {mm_fw} mismatches")
    print(f"   {'✅' if ok_fc else '❌'} car : {mm_fc} mismatches")

# ────────────────────────────────────────────────────────
# 5. Final summary
# ────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

all_checks = [
    ("Internal – primary_storage_allocated sum", results.get("internal_primary_storage_allocated", False)),
    ("Internal – filling_points sum", results.get("internal_filling_points", False)),
    ("Internal – resell sum", results.get("internal_resell", False)),
    ("Cross‑layer – resell = filling + truck + shop", results.get("cross_resell_filling", False)),
    ("Raster – walk reseller IDs all found", results.get("raster_missing_walk", False)),
    ("Raster – car reseller IDs all found", results.get("raster_missing_car", False)),
    ("Raster – walk reseller price match (existing IDs)", results.get("raster_walk_price_match", False)),
    ("Raster – car reseller price match (existing IDs)", results.get("raster_car_price_match", False)),
    ("Raster – filling walk cost match", results.get("raster_filling_walk", False)),
    ("Raster – filling car cost match", results.get("raster_filling_car", False)),
]

pass_count = sum(1 for _, ok in all_checks if ok)
total = len(all_checks)
for desc, ok in all_checks:
    print(f"{'✅' if ok else '❌'} {desc}")

print(f"\n{pass_count}/{total} checks passed.")
if pass_count == total:
    print("🎉 All checks passed. Data is fully consistent.")
else:
    print("⚠️  Some checks failed. Review the details above.")

LOADING DATA
primary_storage_allocated : 19 rows
filling_points            : 375 rows
resell                    : 2,413 rows

1. INTERNAL COMPONENT SUMS
✅ primary_storage_allocated: All 19 rows consistent
✅ filling_points: All 375 rows consistent
✅ resell: All 2413 rows consistent

2. CROSS‑LAYER: RESELL → FILLING
✅ Resell LPG_price = filling LPG_price + cost_truck + cost_res_shop (all rows)

3. RASTER vs VECTOR CONSISTENCY
❌ 275 walk reseller IDs not in resell layer (first 10: [2415, 2417, 2418, 2420, 2421, 2422, 2423, 2424, 2425, 2426])
❌ 275 car reseller IDs not in resell layer (first 10: [2415, 2417, 2418, 2420, 2421, 2422, 2423, 2424, 2425, 2426])

   ── walk reference cost sampling (n=10000) ──
   ⚠️  557 pixels reference IDs missing from resell layer (expected)
   ✅ All remaining pixels match resell LPG_price

   ── car reference cost sampling (n=10000) ──
   ⚠️  567 pixels reference IDs missing from resell layer (expected)
   ✅ All remaining pixels match resell LPG_price

   ──